<img src="Logo.png" width="100" align="left"/> 

# <center> Unit 3 Project </center>
#  <center> Third section : supervised task </center>

In this notebook you will be building and training a supervised learning model to classify your data.

For this task we will be using another classification model "The random forests" model.

Steps for this task: 
1. Load the already clustered dataset 
2. Take into consideration that in this task we will not be using the already added column "Cluster" 
3. Split your data.
3. Build your model using the SKlearn RandomForestClassifier class 
4. classify your data and test the performance of your model 
5. Evaluate the model ( accepted models should have at least an accuracy of 86%). Play with hyper parameters and provide a report about that.
6. Provide evidence on the quality of your model (not overfitted good metrics)
7. Create a new test dataset that contains the testset + an additional column called "predicted_class" stating the class predicted by your random forest classifier for each data point of the test set.

## 1. Load the data and split the data:

In [1]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd 
from sklearn.model_selection import train_test_split
from skopt import BayesSearchCV


In [2]:
# To-Do:  load the data 
df = pd.read_csv('clustered_HepatitisC.csv')
df.head()

,ID,Category,Age,Sex,ALB,ALP,ALT,AST,BIL,CHE,CHOL,CREA,GGT,PROT,cluster
0,1,0,32,1,38.5,52.5,7.7,22.1,7.5,6.93,3.23,106.0,12.1,69.0,3
1,2,0,32,1,38.5,70.3,18.0,24.7,3.9,11.17,4.80,74.0,15.6,76.5,3
2,3,0,32,1,46.9,74.7,36.2,52.6,6.1,8.84,5.20,86.0,33.2,79.3,3
3,4,0,32,1,43.2,52.0,30.6,22.6,18.9,7.33,4.74,80.0,33.8,75.7,3
4,5,0,32,1,39.2,74.1,32.6,24.8,9.6,9.15,4.32,76.0,29.9,68.7,3


In [3]:
# To-Do : keep only the columns to be used : all features except ID, cluster 
# The target here is the Category column 
# Do not forget to split your data (this is a classification task)
# test set size should be 20% of the data 

In [4]:
# Split data into features and target. We drop 'cluster' column as well
X = df.drop(['cluster', 'Category', 'ID'], axis=1)
y = df['Category']
# Check the shape of feature (X) and target (y) sets
print(X.shape)
print(y.shape)


(615, 12)
(615,)


In [5]:
# Split into train and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Verify data splitting by checking the shape
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)



(492, 12)
(492,)
(123, 12)
(123,)


## 2. Building the model and training and evaluate the performance: 

In [6]:
# To-do build the model and train it 
# note that you will be providing explanation about the hyper parameter tuning 
# So you will be iterating a number of times before getting the desired performance 


In [7]:
%pip install scikit-optimize
from skopt.space import Integer
from skopt import BayesSearchCV

Note: you may need to restart the kernel to use updated packages.


In [8]:

rf = RandomForestClassifier(random_state=42, class_weight='balanced'
)

# Define the search space for hyperparameters
search_space = {
    'n_estimators': Integer(10, 100),
    'max_depth': Integer(2, 8),
    'min_samples_leaf': Integer(2, 10)
}

bayes_search = BayesSearchCV(
    estimator=rf,
    search_spaces=search_space,
    n_iter=100,
    cv=3,
    random_state=42,
    n_jobs=-1
)

bayes_search.fit(X_train, y_train)

best_params = bayes_search.best_params_
best_score = bayes_search.best_score_
print("\nBest Hyperparameters found:", best_params)
print("Best Cross-Validation Accuracy:", best_score)

best_model = bayes_search.best_estimator_

/opt/anaconda3/lib/python3.13/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(8), np.int64(2), np.int64(100)] before, using random point [np.int64(5), np.int64(3), np.int64(98)]
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(8), np.int64(2), np.int64(100)] before, using random point [np.int64(6), np.int64(3), np.int64(47)]
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(7), np.int64(2), np.int64(100)] before, using random point [np.int64(5), np.int64(6), np.int64(46)]
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(7), np.int64(2), np.int64(100)] before, using random point [np.int64(5), np.i


Best Hyperparameters found: OrderedDict({'max_depth': 6, 'min_samples_leaf': 3, 'n_estimators': 31})
Best Cross-Validation Accuracy: 0.9491869918699187


In [9]:
y_hat = best_model.predict(X_test)

In [10]:
# To-do : evaluate the model in terms of accuracy and precision 
# Provide evidence that your model is not overfitting 
from sklearn.metrics import precision_score, accuracy_score, classification_report

accuracy = accuracy_score(y_test, y_hat)
precision = precision_score(y_test, y_hat, average='weighted')

print("Accuracy score: ", accuracy)
print("Precision", precision)
print(classification_report(y_test, y_hat))

Accuracy score:  0.9105691056910569
Precision 0.9190565532028947
              precision    recall  f1-score   support

           0       0.92      1.00      0.96        96
           1       1.00      0.67      0.80         3
           2       1.00      0.44      0.62         9
           3       0.57      0.67      0.62         6
           4       1.00      0.67      0.80         9

    accuracy                           0.91       123
   macro avg       0.90      0.69      0.76       123
weighted avg       0.92      0.91      0.90       123



In [11]:
# Compare training and test accuracy
train_accuracy = best_model.score(X_train, y_train)
test_accuracy = best_model.score(X_test, y_test)

print("Train accuracy:", train_accuracy)
print("Test accuracy:", test_accuracy)

Train accuracy: 0.9898373983739838
Test accuracy: 0.9105691056910569


The optimized Random Forest model achieved 98.98% accuracy on the training set and 91.06% accuracy on the test set. The difference between training and test accuracy indicates some overfitting; however, the gap was reduced compared with previous models. The model demonstrates good generalization performance on unseen data. Hyperparameter optimization using Bayesian Optimization and class balancing improved the robustness of the classifier, particularly for minority classes.

> Hint : A Perfect accuracy on the train set suggest that we have an overfitted model So the student should be able to provide a detailed table about the hyper parameters / parameters tuning with a good conclusion stating that the model has at least an accuracy of 86% on the test set without signs of overfitting  

## 3. Create the summary test set with the additional predicted class column: 
In this part you need to add the predicted class as a column to your test dataframe and save this one 

In [23]:
# To-Do : create the complete test dataframe : it should contain all the feature column + the actual target and the ID as well  
test_df = X_test.copy()
test_df["Category"] = df['Category']
test_df["cluster"] = df["cluster"]
test_df["ID"] = df.loc[X_test.index, "ID"]


In [24]:
test_df.head()

,Age,Sex,ALB,ALP,ALT,AST,BIL,CHE,CHOL,CREA,GGT,PROT,Category,cluster,ID
248,55,1,28.1,65.5,16.6,17.5,2.8,5.58,4.39,65.0,26.2,62.4,0,3,249
365,39,0,31.4,106.0,16.6,17.0,2.4,5.95,5.30,68.0,22.9,72.3,0,3,366
432,48,0,43.7,50.1,17.3,26.3,8.1,8.15,5.38,64.0,13.4,73.1,0,3,433
610,62,0,32.0,416.6,5.9,110.3,50.0,5.57,6.30,55.7,650.9,68.5,4,2,611
132,44,1,35.5,81.7,27.5,29.5,6.4,8.81,6.65,83.0,24.1,68.0,0,3,133


In [25]:
# To-Do : Add the predicted_class column 
test_df["Predicted_class"] = y_hat

In [26]:
test_df.head()
test_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 123 entries, 248 to 336
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Age              123 non-null    int64  
 1   Sex              123 non-null    int64  
 2   ALB              123 non-null    float64
 3   ALP              123 non-null    float64
 4   ALT              123 non-null    float64
 5   AST              123 non-null    float64
 6   BIL              123 non-null    float64
 7   CHE              123 non-null    float64
 8   CHOL             123 non-null    float64
 9   CREA             123 non-null    float64
 10  GGT              123 non-null    float64
 11  PROT             123 non-null    float64
 12  Category         123 non-null    int64  
 13  cluster          123 non-null    int64  
 14  ID               123 non-null    int64  
 15  Predicted_class  123 non-null    int64  
dtypes: float64(10), int64(6)
memory usage: 16.3 KB


> Make sure you have 16 column in this test set  

In [27]:
# Save the test set 
test_df.to_csv("test_summary.csv")